# **LightGBM**



LightGBM — это библиотека градиентного бустинга по деревьям решений, которая:

ускоряет обучение за счёт гистограмм и специальных трюков (GOSS, EFB);

строит деревья leaf-wise (по «наиболее выгодному листу»), а не уровень-за-уровнем, как XGBoost;

умеет нативно работать с категориальными признаками (их можно не one-hot-ить);

хорошо масштабируется на большие датасеты.

In [ ]:
!pip install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


Таргет

In [ ]:
def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # Будущая цена
    df['Close_fwd'] = df['Close'].shift(-horizon)

    # Доходность по направлению сигнала
    ret_long = (df['Close_fwd'] - df['Close']) / df['Close']
    ret_short = (df['Close'] - df['Close_fwd']) / df['Close']

    ret = np.where(
        df['EntrySignal'] > 0, ret_long,
        np.where(df['EntrySignal'] < 0, ret_short, 0.0)
    )

    df['ret_H'] = ret

    # GoodTrade: есть сигнал и через H баров прибыль > 0
    df['GoodTrade'] = ((df['EntrySignal'] != 0) & (df['ret_H'] > 0)).astype(int)

    # убираем последние horizon строк (там нет Close_fwd)
    df = df.iloc[:-horizon]

    return df

In [ ]:
df = pd.read_csv("Brent.csv")
df = add_goodtrade_target(df, horizon=20)

In [ ]:
cols_nan = ["AddOn_Anchor_Level", "AddOn_Anchor_IsUp", "AddOn_Size_Pct"]

df = df.dropna(subset=cols_nan).reset_index(drop=True)

In [ ]:
# GoodProb: "насколько хороша сделка" по факту

# 1) Обрезаем хвосты доходности
low_q = df['ret_H'].quantile(0.01)
high_q = df['ret_H'].quantile(0.99)
ret_capped = df['ret_H'].clip(low_q, high_q)

# 2) Нормируем в [0, 1]
df['GoodProb'] = (ret_capped - ret_capped.min()) / (ret_capped.max() - ret_capped.min())

print(df[['ret_H', 'GoodProb']].head())


     ret_H  GoodProb
0  0.00000  0.510012
1  0.00000  0.510012
2  0.00000  0.510012
3  0.00632  0.699138
4  0.00000  0.510012


In [ ]:
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered,Close_fwd,ret_H,GoodTrade,GoodProb
0,2015-10-26 19:00:00,47.76,48.00,47.70,47.86,48.15240,48.12767,48.01926,1,0,...,1,47.70,0.0,0.3,1,1,47.08,0.00000,0,0.510012
1,2015-10-26 20:00:00,47.86,47.91,47.54,47.65,48.16106,48.10359,47.94841,0,0,...,0,47.70,0.0,0.3,0,0,47.10,0.00000,0,0.510012
2,2015-10-26 21:00:00,47.65,47.82,47.52,47.57,48.16367,48.05876,47.91373,0,0,...,0,47.70,0.0,0.3,0,0,47.16,0.00000,0,0.510012
3,2015-10-26 22:00:00,47.57,47.59,47.43,47.47,48.16070,48.00954,47.90098,0,0,...,0,47.70,0.0,0.3,0,0,47.17,0.00632,1,0.699138
4,2015-10-26 23:00:00,47.47,47.51,47.31,47.31,48.14334,47.98022,47.86578,0,0,...,0,47.70,0.0,0.3,0,0,47.10,0.00000,0,0.510012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36210,2025-10-20 18:00:00,60.73,61.04,60.51,60.60,61.31318,61.20603,61.06995,0,0,...,1,63.53,0.0,0.3,0,0,61.65,0.00000,0,0.510012
36211,2025-10-20 19:00:00,60.68,60.79,60.63,60.73,61.29409,61.16590,60.96996,0,0,...,0,63.53,0.0,0.3,0,0,62.46,0.00000,0,0.510012
36212,2025-10-20 20:00:00,60.73,61.06,60.66,60.98,61.28646,61.13203,60.92697,0,0,...,0,63.53,0.0,0.3,0,0,62.35,0.00000,0,0.510012
36213,2025-10-20 21:00:00,60.98,61.14,60.92,60.96,61.27366,61.06178,60.89658,1,0,...,0,63.53,0.0,0.3,0,0,62.36,0.00000,0,0.510012


Кодируем признаки

In [ ]:
# 1. Заполняем пропуски и делаем категорию
df['EntryReason'] = df['EntryReason'].fillna('None').astype('category')

# 2. Смотрим маппинг кодов (для контроля, можно один раз глянуть)
entryreason_mapping = dict(enumerate(df['EntryReason'].cat.categories))
print("Mapping EntryReason:", entryreason_mapping)

# 3. Перезаписываем EntryReason числовыми кодами
df['EntryReason'] = df['EntryReason'].cat.codes.astype('int16')

# 4. Удаляем лишние текстовые столбцы, если они есть
df = df.drop(columns=['EntryReason_raw', 'EntryReason_code'], errors='ignore')

# 5. Проверяем, что не осталось других строк/категорий
print("Нечисловые колонки:",
      df.select_dtypes(include=['object', 'category']).columns.tolist())


Mapping EntryReason: {0: 'None', 1: 'saucer', 2: 'three_color', 3: 'zero_cross'}
Нечисловые колонки: ['DateTime']


Подготовка признаков

Подготовка даннных

In [ ]:
import numpy as np

def prepare_train_data_reg(df, target_col='GoodProb', signals_only=True):
    df = df.copy()

    # Берём только бары, где есть сигнал (мы оцениваем качество входов)
    if signals_only:
        df = df[df['EntrySignal'] != 0]

    # Колонки, которые точно НЕ должны стать признаками
    drop_cols = ['DateTime', 'Close_fwd', 'ret_H', 'GoodTrade']
    drop_cols = [c for c in drop_cols if c in df.columns]

    # Кандидаты в признаки
    candidate_cols = [
        c for c in df.columns
        if c not in drop_cols + [target_col]
    ]

    # Оставляем только числовые признаки
    feature_cols = df[candidate_cols].select_dtypes(include=[np.number]).columns.tolist()

    X = df[feature_cols].astype('float32').values
    y = df[target_col].values.astype('float32')  # теперь это не 0/1, а [0..1]

    return X, y, feature_cols, df


In [ ]:
X, y, feature_cols, df_signals = prepare_train_data_reg(df, target_col='GoodProb')

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("Shapes:", X_train.shape, X_test.shape)
print("y_train range:", y_train.min(), "→", y_train.max())



Shapes: (1430, 31) (358, 31)
y_train range: 0.0 → 1.0


Баланс классов

In [ ]:
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
print("GoodTrade=1:", pos, "GoodTrade=0:", neg, "neg/pos =", neg / pos)

class_weight = {0: 1.0, 1: neg / pos}


GoodTrade=1: 701 GoodTrade=0: 729 neg/pos = 1.0399429386590584


Модель

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

lgbm_reg = LGBMRegressor(
    n_estimators=500,
    num_leaves=31,
    learning_rate=0.05,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    n_jobs=-1,
    random_state=42
)

lgbm_reg.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='l2',   # MSE на валидации
)

y_pred = lgbm_reg.predict(X_test)

print("R^2:", r2_score(y_test, y_pred))
print("Spearman corr:", spearmanr(y_test, y_pred))


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070100

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd

df_eval = df_signals.iloc[split_idx:].copy()  # та же выборка, что X_test / y_test
df_eval['GoodProb_true'] = y_test
df_eval['GoodProb_pred'] = y_pred

# 5 квантилей по предсказанию модели
df_eval['bucket'] = pd.qcut(df_eval['GoodProb_pred'], 5, labels=False)

print(
    df_eval.groupby('bucket')['ret_H'].mean()
)


bucket
0    0.001282
1    0.000706
2   -0.001457
3   -0.000672
4   -0.002434
Name: ret_H, dtype: float64
